# Local Outlier Factor (LOF)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

## Load data and split

In [ ]:
data = pd.read_csv('../data/labeled.csv', parse_dates=['timestamp'])
split_point = int(len(data) * 0.8)
train_data = data.iloc[:split_point]
test_data = data.iloc[split_point:]

feature_columns = ['memory_pct', 'roll_mean_1h', 'roll_std_1h', 'roll_mean_24h',
                   'diff_1', 'diff_6', 'hour', 'minute', 'time_of_day', 'day_of_week', 'is_weekend']

X_train = train_data[feature_columns].values
X_test = test_data[feature_columns].values
y_test = test_data['label'].values

print('Anomalies in test:', y_test.sum())

## Train LOF

In [ ]:
lof_model = LocalOutlierFactor(n_neighbors=50, contamination=0.005, novelty=True)
lof_model.fit(X_train)

lof_predictions = (lof_model.predict(X_test) == -1).astype(int)
lof_scores = -lof_model.score_samples(X_test)

print('Flagged points:', lof_predictions.sum())

## Plot detections

In [ ]:
timestamps = test_data['timestamp'].values
memory = test_data['memory_pct'].values

plt.figure(figsize=(14, 4))
plt.plot(timestamps, memory, color='lightblue')
plt.scatter(timestamps[y_test == 1], memory[y_test == 1], color='red', marker='x', label='True anomaly')
plt.scatter(timestamps[lof_predictions == 1], memory[lof_predictions == 1], color='purple', s=10, label='LOF flagged')
plt.title('Local Outlier Factor')
plt.ylabel('Memory %')
plt.legend()
plt.show()

## Evaluation

In [ ]:
print(classification_report(y_test, lof_predictions, target_names=['Normal','Anomaly'], zero_division=0))
print('Confusion matrix:')
print(confusion_matrix(y_test, lof_predictions))
print('ROC-AUC:', round(roc_auc_score(y_test, lof_scores), 4))